<!-- NB_DOC_INTRO_V1 -->
# NLP — NER avec BiLSTM-CRF (archi historique)

> 📚 **Doc thematique** : [docs/05_NLP.md](docs/05_NLP.md) (NLP)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**BiLSTM-CRF** = archi reference pre-BERT pour NER (2015-2018). Toujours pertinente pour comprendre **pourquoi** la **CRF** layer aide a la coherence des sequences (eviter des sequences invalides type 'I-PER apres O').

Ce notebook explique l'architecture + execute une **demo simplifiee en PyTorch** (sans CRF reelle pour eviter dep externe, mais avec viterbi-like decoding). Pour CRF complete : `torchcrf` ou `tensorflow_addons`.

Pour NER moderne (Transformers pre-entraine) : [NLP_NER.ipynb](./NLP_NER.ipynb).

## Plan

1. Architecture BiLSTM-CRF expliquee
2. Pourquoi CRF (transition matrix)
3. Demo simplifiee en PyTorch (BiLSTM sans CRF + viterbi-like greedy decode)
4. Comparaison avec / sans CRF
5. Limites de l'archi pre-BERT
6. Alternative moderne (Transformers + CRF head)
7. Pieges + References


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import warnings
warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Architecture BiLSTM-CRF

```
Input tokens : [The, dog, eats, fish]
                ↓
        [Embedding layer]      → vecteurs denses (50-300 dims)
                ↓
        [BiLSTM (forward + backward)]   → context bidirectionnel
                ↓
        [Linear layer]         → scores par tag (B-PER, I-PER, O, ...)
                ↓
        [CRF layer]            → contraint la sequence de tags valides
                ↓
        Output : [O, B-ANIMAL, O, B-FOOD]
```

### Pourquoi CRF ?

Sans CRF, le modele predit **independamment** chaque tag → peut produire des sequences impossibles :
- `O → I-PER` (impossible : I doit suivre B ou I de la meme entite)
- `B-PER → I-LOC` (impossible : transition incoherente)

**CRF** apprend une **matrice de transition** entre tags → favorise les sequences valides. Decode via **Viterbi**.


## 2. Demo simplifiee — BiLSTM tagger

In [ ]:
# Petit dataset jouet : phrases avec tags par token
# (en pratique : CoNLL-2003 ou WNUT)
sentences = [
    [("Apple", "B-ORG"), ("is", "O"), ("based", "O"), ("in", "O"), ("California", "B-LOC")],
    [("Tim", "B-PER"), ("Cook", "I-PER"), ("works", "O"), ("at", "O"), ("Apple", "B-ORG")],
    [("Paris", "B-LOC"), ("is", "O"), ("in", "O"), ("France", "B-LOC")],
    [("The", "O"), ("dog", "O"), ("eats", "O"), ("fish", "O")],
    [("Microsoft", "B-ORG"), ("CEO", "O"), ("Satya", "B-PER"), ("Nadella", "I-PER"), ("spoke", "O")],
]

# Build vocab + tag set
words = sorted({w for s in sentences for w, _ in s})
tags = sorted({t for s in sentences for _, t in s})
word2idx = {w: i+2 for i, w in enumerate(words)}    # 0 = PAD, 1 = UNK
word2idx["<PAD>"] = 0; word2idx["<UNK>"] = 1
tag2idx = {t: i for i, t in enumerate(tags)}
idx2tag = {i: t for t, i in tag2idx.items()}
print(f"Vocab : {len(word2idx)}, Tags : {tags}")

In [ ]:
# Encoder dataset
def encode_sentence(sentence):
    word_ids = [word2idx.get(w, 1) for w, _ in sentence]
    tag_ids = [tag2idx[t] for _, t in sentence]
    return word_ids, tag_ids

# === BiLSTM model ===
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, n_tags, embed_dim=32, hidden=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden // 2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden, n_tags)

    def forward(self, x):
        emb = self.embed(x)
        lstm_out, _ = self.lstm(emb)
        return self.fc(lstm_out)   # logits shape (batch, seq_len, n_tags)

model = BiLSTMTagger(len(word2idx), len(tag2idx))
print(f"Params : {sum(p.numel() for p in model.parameters()):,}")

## 3. Training (sans CRF, juste BiLSTM + cross-entropy)

In [ ]:
# Padding : on prend max_len = longest sentence
max_len = max(len(s) for s in sentences)

def collate(batch):
    word_ids_b, tag_ids_b = zip(*batch)
    X = torch.zeros(len(batch), max_len, dtype=torch.long)
    Y = -100 * torch.ones(len(batch), max_len, dtype=torch.long)   # -100 = ignored par CE
    for i, (w_ids, t_ids) in enumerate(batch):
        X[i, :len(w_ids)] = torch.tensor(w_ids)
        Y[i, :len(t_ids)] = torch.tensor(t_ids)
    return X, Y

dataset = [encode_sentence(s) for s in sentences]
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

for epoch in range(100):
    X, Y = collate(dataset)
    logits = model(X)  # (B, L, T)
    loss = criterion(logits.permute(0, 2, 1), Y)  # CE attend (B, T, L)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}  loss = {loss.item():.4f}")

In [ ]:
# Inference sur une phrase
def predict(model, sentence_words):
    model.eval()
    word_ids = torch.tensor([[word2idx.get(w, 1) for w in sentence_words]], dtype=torch.long)
    with torch.no_grad():
        logits = model(word_ids)
    preds = logits.argmax(-1)[0].tolist()
    return [(w, idx2tag[p]) for w, p in zip(sentence_words, preds)]

# Sur une phrase d'entrainement (devrait etre quasi-parfait)
print("Train sentence :")
for w, t in predict(model, ["Apple", "is", "based", "in", "California"]):
    print(f"  {w:12s} → {t}")

print("\nNouvelle sentence :")
for w, t in predict(model, ["Microsoft", "is", "in", "Seattle"]):
    print(f"  {w:12s} → {t}")

## 4. Pourquoi ajouter une CRF layer ?

Sans CRF, chaque tag est predit **independamment** : le modele peut faire :
- `O → I-LOC` (impossible : I doit suivre B-LOC ou I-LOC)
- `B-PER → I-ORG` (changement d'entite sans B)

Avec CRF, on apprend une **matrice de transitions** entre tags. Au decode, on cherche la **sequence globalement la plus probable** via **Viterbi** (programmation dynamique).

```python
# Code de reference avec torchcrf
# pip install pytorch-crf
from torchcrf import CRF

class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, n_tags, embed_dim=32, hidden=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden // 2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden, n_tags)
        self.crf = CRF(n_tags, batch_first=True)

    def forward(self, x, tags=None, mask=None):
        emb = self.embed(x)
        lstm_out, _ = self.lstm(emb)
        logits = self.fc(lstm_out)
        if tags is not None:
            # Training : CRF loss
            return -self.crf(logits, tags, mask=mask, reduction="mean")
        else:
            # Inference : viterbi decode
            return self.crf.decode(logits, mask=mask)
```

## 5. Alternative moderne : Transformers + CRF head

En 2024-2025, on remplace souvent le BiLSTM par un **Transformer pre-entraine** (BERT, CamemBERT, ...). La CRF reste optionnelle (parfois gain marginal).

```python
from transformers import AutoModel, AutoTokenizer

class BERTNERWithCRF(nn.Module):
    def __init__(self, model_name, n_tags):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_tags)
        self.crf = CRF(n_tags, batch_first=True)
    # ... idem mais avec bert(input_ids).last_hidden_state au lieu de LSTM
```

## 6. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| BiLSTM sans CRF sur dataset avec transitions complexes | Ajouter CRF (peu de cout, gain ~ 1-3% F1) |
| CRF + BERT (double overhead) | Souvent inutile : BERT seul suffit |
| Padding sans mask | CRF compte les pad tokens — toujours `mask=` |
| Pas `ignore_index=-100` dans CE loss | Compte le padding (faux) |
| Embedding dim trop petit (10-20) | 32-128 min pour LSTM, 768 pour BERT |
| Pas de cleaning (case sensitivity) | Souvent meilleur en lowercase + features case-aware |

## 7. References

- BiLSTM-CRF original : Huang et al. 2015 — https://arxiv.org/abs/1508.01991
- *Neural Architectures for Named Entity Recognition* (Lample et al. 2016)
- pytorch-crf : https://pytorch-crf.readthedocs.io/
- BERT-based NER (HuggingFace) : https://huggingface.co/docs/transformers/tasks/token_classification

### Voir aussi (collection)
- [NLP_NER.ipynb](NLP_NER.ipynb) — NER moderne (spaCy + Transformers)
- [NLP_Transformers.ipynb](NLP_Transformers.ipynb) — bases Transformers
- [DL_PyTorch.ipynb](DL_PyTorch.ipynb)
